## 📦 ADIM 1 — Gerekli kütüphaneleri kur

In [1]:
!pip install -q gradio groq langchain langchain-community \
    chromadb sentence-transformers tiktoken
print('✅ Kurulum tamamlandı!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/2

## 📁 ADIM 2 — Google Drive'ı bağla ve dosyaları kopyala

**Drive'ımda şu yapı var**
```
MyDrive/
  firat_chatbot/
    cikti_txt/   ← Tüm .txt dosyaları buraya
    data/
      train.jsonl
```


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive bağlandı!')

Mounted at /content/drive
✅ Google Drive bağlandı!


In [3]:
import os, shutil

# Drive'daki klasör yolları
DRIVE_TXT_FOLDER = '/content/drive/MyDrive/firat_chatbot/cikti_txt'
DRIVE_JSONL_FILE = '/content/drive/MyDrive/firat_chatbot/data/train.jsonl'

# Colab'da çalışma dizinleri
LOCAL_TXT_FOLDER = '/content/cikti_txt'
LOCAL_DATA_FOLDER = '/content/data'

# Klasörleri oluştur
os.makedirs(LOCAL_TXT_FOLDER, exist_ok=True)
os.makedirs(LOCAL_DATA_FOLDER, exist_ok=True)

# Dosyaları kopyala
if os.path.exists(DRIVE_TXT_FOLDER):
    for fname in os.listdir(DRIVE_TXT_FOLDER):
        if fname.endswith('.txt'):
            shutil.copy(os.path.join(DRIVE_TXT_FOLDER, fname),
                        os.path.join(LOCAL_TXT_FOLDER, fname))
    txt_count = len([f for f in os.listdir(LOCAL_TXT_FOLDER) if f.endswith('.txt')])
    print(f'✅ {txt_count} adet TXT dosyası kopyalandı.')
else:
    print('⚠️  Drive klasörü bulunamadı. Lütfen DRIVE_TXT_FOLDER yolunu kontrol edin.')

if os.path.exists(DRIVE_JSONL_FILE):
    shutil.copy(DRIVE_JSONL_FILE, '/content/data/train.jsonl')
    print('✅ train.jsonl kopyalandı.')
else:
    print('⚠️  train.jsonl bulunamadı (isteğe bağlı).')

✅ 40 adet TXT dosyası kopyalandı.
✅ train.jsonl kopyalandı.


## 🔑 ADIM 3 — Google API anahtarını gir

In [ ]:
from groq import Groq

GROQ_API_KEY = ''

client = Groq(api_key=GROQ_API_KEY)

# Test
test = client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[{'role': 'user', 'content': 'Merhaba, çalışıyor musun?'}]
)
print('✅ Groq API bağlı!')
print('Test:', test.choices[0].message.content[:100])

✅ Groq API bağlı!
Test: Merhaba! Evet, çalışıyorum. Sivil bir yardımcı program olarak tasarlandım ve sizlere yardımcı olmak,


## 🧠 ADIM 4 — Belge tabanını oluştur (RAG vektör veritabanı)

Bu adım TXT dosyalarını okuyup semantik arama için vektörlere dönüştürür.
~2-3 dakika sürebilir.

In [5]:
!pip install -q langchain langchain-community langchain-huggingface chromadb sentence-transformers
print("✅ Kurulum bitti!")

✅ Kurulum bitti!


In [6]:
!pip install -q langchain-text-splitters

In [7]:
!pip install -q langchain-core

In [8]:
import json, os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = []
txt_folder = LOCAL_TXT_FOLDER

isim_haritasi = {
    'aktsuygulama': 'AKTS Uygulama Yönergesi',
    'azamiogrenimsuresi': 'Azami Öğrenim Süresi',
    'bitirmeprojesi': 'Bitirme Projesi Yönergesi',
    'ciftanadalveyandal': 'Çift Anadal ve Yandal',
    'danismakurallari': 'Danışma Kurulları Yönergesi',
    'disiplinlerarasilisans': 'Disiplinlerarası Lisans',
    'dortluksistem': 'Dörtlük Sistem',
    'egitimkomisyonu': 'Eğitim Komisyonu',
    'ekdersodeme': 'Ek Ders Ödeme',
    'ekdersyonetim': 'Ek Ders Yönetimi',
    'ekmadde': 'Ek Madde',
    'enbasarilitezodulu': 'En Başarılı Tez Ödülü',
    'etikkurul': 'Etik Kurul',
    'fu_bilgi': 'Fırat Üniversitesi Genel Bilgi',
    'hakligecerlinedenler': 'Haklı Geçerli Nedenler',
    'ickontrolizleme': 'İç Kontrol İzleme',
    'ickontrolsistemi': 'İç Kontrol Sistemi',
    'isagligi': 'İş Sağlığı',
    'lisansustu': 'Lisansüstü Yönetmelik',
    'mevzuat': 'Mevzuat',
    'mufredatguncelleme': 'Müfredat Güncelleme',
    'ogrencisenatosu': 'Öğrenci Senatosu',
    'onlisansvelisansegitimyonetmeligi': 'Önlisans ve Lisans Eğitim Yönetmeliği',
    'onlisansvelisansyonetmeligi': 'Önlisans ve Lisans Yönetmeliği',
    'onmalikontrol': 'Ön Mali Kontrol',
    'ortakders': 'Ortak Ders',
    'ozelogrenci': 'Özel Öğrenci',
    'pedagojikformasyon': 'Pedagojik Formasyon',
    'saglikenstitusu': 'Sağlık Enstitüsü',
    'sahipsizhayvan': 'Sahipsiz Hayvan Yönergesi',
    'senatoesaslari': 'Senato Esasları',
    'sinavyonetmeligi': 'Sınav Yönetmeliği',
    'tezodulu': 'Tez Ödülü',
    'tipkurallari': 'Tıp Kuralları',
    'turkceuygulama': 'Türkçe Uygulama',
    'uluslarasiogrenci': 'Uluslararası Öğrenci',
    'uluslarasiogrencisecme': 'Uluslararası Öğrenci Seçme',
    'yabancidilegitimi': 'Yabancı Dil Eğitimi',
    'yataygecis': 'Yatay Geçiş Yönetmeliği',
    'yazokulu': 'Yaz Okulu',
}

for fname in os.listdir(txt_folder):
    if not fname.endswith('.txt'):
        continue
    key = fname.replace('.txt', '')
    baslik = isim_haritasi.get(key, key)
    with open(os.path.join(txt_folder, fname), 'r', encoding='utf-8') as f:
        icerik = f.read().strip()
    if icerik:
        documents.append(Document(page_content=icerik, metadata={'kaynak': baslik}))

print(f'📄 {len(documents)} belge yüklendi.')

# train.jsonl ekle
jsonl_path = '/content/data/train.jsonl'
if os.path.exists(jsonl_path):
    ekle = 0
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line.strip())
                msgs = item.get('messages', [])
                soru = next((m['content'] for m in msgs if m['role']=='user'), '')
                cevap = next((m['content'] for m in msgs if m['role']=='assistant'), '')
                if soru and cevap:
                    documents.append(Document(
                        page_content=f'Soru: {soru}\nCevap: {cevap}',
                        metadata={'kaynak': 'Hazır S/C'}
                    ))
                    ekle += 1
            except: pass
    print(f'🔗 {ekle} soru-cevap eklendi.')

# Parçala
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f'✂️  {len(chunks)} parça oluşturuldu.')

# Embedding
print('⏳ Embedding yükleniyor (~1-2 dk)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cpu'}
)

# Vektör DB
print('⏳ Vektör veritabanı oluşturuluyor...')
vectordb = Chroma.from_documents(chunks, embedding=embeddings, persist_directory='/content/chroma_db')
print(f'✅ Hazır! ({vectordb._collection.count()} vektör)')

📄 40 belge yüklendi.
🔗 253 soru-cevap eklendi.
✂️  1304 parça oluşturuldu.
⏳ Embedding yükleniyor (~1-2 dk)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Vektör veritabanı oluşturuluyor...
✅ Hazır! (1304 vektör)


## 🤖 ADIM 5 — RAG + Chatbot fonksiyonlarını tanımla

In [9]:
from groq import Groq
client = Groq(api_key=GROQ_API_KEY)

def soru_sor(soru: str) -> str:
    if not soru.strip():
        return 'Lütfen bir soru yazın.'

    ilgili_docs = vectordb.similarity_search(soru, k=5)
    parcalar = [f"[{d.metadata.get('kaynak','?')}]\n{d.page_content}" for d in ilgili_docs]
    baglam = '\n\n---\n\n'.join(parcalar)

    yanit = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {
                'role': 'system',
                'content': '''Sen Fırat Üniversitesi danışman asistanısın.
Sadece verilen belgelere dayanarak Türkçe cevap ver.
Emin olmadığın konularda "Bu konuda ilgili birime danışın" de.'''
            },
            {
                'role': 'user',
                'content': f'''Belgeler:\n{baglam}\n\nSoru: {soru}'''
            }
        ],
        temperature=0.3,
        max_tokens=500
    )
    return yanit.choices[0].message.content.strip()

# Test
print(soru_sor('Yatay geçiş için not ortalaması kaç olmalıdır?'))
print('✅ Chatbot hazır!')

Yatay geçiş için not ortalaması, öğrencinin kayıtlı olduğu programda bitirmiş olduğu dönemle ilgili GNO'sunun, dörtlük sistemde en az 2.50, yüzlük sistemde ise en az 65 olması gerekir.
✅ Chatbot hazır!


## 🎨 ADIM 6 — Arayüzü başlat (Gradio)

In [10]:
import gradio as gr

# Hazır örnek sorular
ORNEK_SORULAR = [
    '📋 Yatay geçiş koşulları nelerdir?',
    '📅 Yaz okulu nasıl açılır?',
    '🎓 Çift anadal programına nasıl başvururum?',
    '📝 Özel öğrenci kimdir?',
    '⏱️ Azami öğrenim süresi ne kadardır?',
    '📊 Dörtlük not sistemi nasıl hesaplanır?',
    '🌍 Uluslararası öğrenci nasıl kayıt yaptırır?',
    '📖 Bitirme projesi koşulları nelerdir?',
    'Fırat Üniversitesi ne zaman kurulmuştur?',
]

def chatbot_yanit(mesaj, gecmis):
    """Gradio chat interface handler."""
    mesaj = mesaj.replace('📋 ', '').replace('📅 ', '').replace('🎓 ', '') \
                 .replace('📝 ', '').replace('⏱️ ', '').replace('📊 ', '') \
                 .replace('🌍 ', '').replace('📖 ', '')
    yanit = soru_sor(mesaj)
    return yanit

# CSS ve tema
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Nunito:wght@400;600;700;800&display=swap');

* { font-family: 'Nunito', sans-serif !important; }

.gradio-container {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%) !important;
    min-height: 100vh;
}

#baslik-kutu {
    background: rgba(255,255,255,0.05);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 20px;
    padding: 28px 36px;
    margin-bottom: 24px;
    text-align: center;
    backdrop-filter: blur(10px);
}

#logo-text {
    font-size: 42px;
    font-weight: 800;
    background: linear-gradient(90deg, #e94560, #0f3460, #533483);
    background-size: 200% auto;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    animation: shimmer 3s linear infinite;
}

@keyframes shimmer {
    to { background-position: 200% center; }
}

.furow { color: #a0c4ff; font-size: 15px; margin-top: 6px; }

#chat-kutu {
    background: rgba(255,255,255,0.04) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    border-radius: 16px !important;
}

.message.bot {
    background: rgba(14, 52, 96, 0.7) !important;
    border: 1px solid rgba(100, 160, 255, 0.2) !important;
    border-radius: 12px !important;
    color: #e8f4f8 !important;
}

.message.user {
    background: rgba(233, 69, 96, 0.3) !important;
    border: 1px solid rgba(233, 69, 96, 0.3) !important;
    border-radius: 12px !important;
    color: #fff !important;
}

textarea, input[type='text'] {
    background: rgba(255,255,255,0.07) !important;
    border: 1px solid rgba(255,255,255,0.2) !important;
    border-radius: 12px !important;
    color: #fff !important;
    font-size: 15px !important;
}

button.primary {
    background: linear-gradient(135deg, #e94560, #c0392b) !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    letter-spacing: 0.5px !important;
}

button.secondary {
    background: rgba(255,255,255,0.08) !important;
    border: 1px solid rgba(255,255,255,0.2) !important;
    border-radius: 8px !important;
    color: #cce0ff !important;
    font-size: 13px !important;
}

button.secondary:hover {
    background: rgba(233, 69, 96, 0.3) !important;
    border-color: rgba(233, 69, 96, 0.5) !important;
}

label, .label-wrap { color: #a0c4ff !important; }
.footer { display: none !important; }
"""

with gr.Blocks(
    css=CUSTOM_CSS,
    title='FÜ Danışman Chatbot',
    theme=gr.themes.Base(
        primary_hue='red',
        secondary_hue='blue',
        neutral_hue='slate',
    )
) as demo:

    # --- Başlık ---
    with gr.Row():
        gr.HTML("""
        <div id="baslik-kutu">
          <div id="logo-text">🎓 FÜ Danışman Asistan</div>
          <div class="furow">Fırat Üniversitesi · Danışman Kolu · Yapay Zeka Destekli</div>
        </div>
        """)

    # --- Ana bölüm ---
    with gr.Row():

        # Sol panel: örnek sorular
        with gr.Column(scale=1):
            gr.Markdown('### 💡 Örnek Sorular', elem_classes='label-wrap')
            ornek_butonlari = []
            for soru in ORNEK_SORULAR:
                btn = gr.Button(soru, variant='secondary', size='sm')
                ornek_butonlari.append(btn)

            gr.HTML('<br>')
            gr.Markdown('### ℹ️ Hakkında', elem_classes='label-wrap')
            gr.HTML("""
            <div style="color:#90b4d4; font-size:13px; line-height:1.7;">
              <b style="color:#cce0ff;">Kapsam:</b> Üniversite yönetmelikleri,
              yönergeler, sınav ve geçiş kuralları,
              öğrenci hakları ve akademik süreçler.<br><br>
              <b style="color:#cce0ff;">Kaynak:</b> Fırat Üniversitesi resmi
              belgeleri (46 yönetmelik & yönerge).<br><br>
              <b style="color:#cce0ff;">Not:</b> Kesin bilgi için ilgili
              üniversite birimiyle iletişime geçin.
            </div>
            """)

        # Sağ panel: chat
        with gr.Column(scale=3, elem_id='chat-kutu'):
            chatbot = gr.Chatbot(
                label='',
                height=520,
                show_label=False,
                avatar_images=(
                    None,
                    'https://i.imgur.com/8Km9tLL.png'
                ),
                bubble_full_width=False,
            )

            with gr.Row():
                kullanici_girdisi = gr.Textbox(
                    placeholder='Sorunuzu buraya yazın... (örn: yatay geçiş şartları nelerdir?)',
                    label='',
                    show_label=False,
                    lines=2,
                    scale=5,
                )
                with gr.Column(scale=1):
                    gonder_btn = gr.Button('Gönder ➤', variant='primary', size='lg')
                    temizle_btn = gr.Button('Temizle 🗑️', variant='secondary', size='sm')

    # --- Event bağlantıları ---
    def gonder(mesaj, gecmis):
        if not mesaj.strip():
            return gecmis, ''
        yanit = chatbot_yanit(mesaj, gecmis)
        gecmis = gecmis + [[mesaj, yanit]]
        return gecmis, ''

    def ornek_sec(soru):
        return soru

    # Gönder butonları
    gonder_btn.click(
        fn=gonder,
        inputs=[kullanici_girdisi, chatbot],
        outputs=[chatbot, kullanici_girdisi]
    )
    kullanici_girdisi.submit(
        fn=gonder,
        inputs=[kullanici_girdisi, chatbot],
        outputs=[chatbot, kullanici_girdisi]
    )
    temizle_btn.click(fn=lambda: ([], ''), outputs=[chatbot, kullanici_girdisi])

    # Örnek soru butonları
    for btn in ornek_butonlari:
        btn.click(
            fn=lambda s: s,
            inputs=[btn],
            outputs=[kullanici_girdisi]
        )

print('🚀 Arayüz başlatılıyor...')
demo.launch(share=True, debug=False)

/tmp/ipykernel_16085/1993522913.py:114: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_16085/1993522913.py:114: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_16085/1993522913.py:160: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_16085/1993522913.py:160: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_

🚀 Arayüz başlatılıyor...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b96661da8b3bd5632.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import shutil
shutil.copytree('/content/chroma_db',
                '/content/drive/MyDrive/firat_chatbot/chroma_db',
                dirs_exist_ok=True)
print('✅ Vektör DB Drive\'a kaydedildi!')

✅ Vektör DB Drive'a kaydedildi!
